# NB14: news signal audit, and the corrected signal file the dashboard uses

Before the news workstream could be wired into the dashboard, every news output produced on this
project was re-checked against the APIs that made it. Two of the three turned out to need
correcting, so this notebook records the audit and writes one trustworthy file.

**What was found on disk**

| Output | Author | Sample | Reported result | Verdict |
|---|---|---|---|---|
| `nb08_spine_news_signals.csv` | Viktor, NB08 | 96 | 92 of 96 have coverage | **Not usable.** Query defect, see section 2 |
| `nb09_guardian_signals.csv` | Vishal, NB09 | 96 | 2 verified | **Stale.** Predates the verification hardening, see section 3 |
| `guardian_results.csv`, `gnews_results.csv`, NB07 NewsAPI | NB07 feasibility | 72 | 2, 3 and 5 hits | Method sound but unverified, all hits are collisions |

**The corrected position.** Across every API tried (Guardian, GNews, NewsAPI, and GDELT which never
ran because the IP was blocked), and across both samples, **no company has verifiable, company
specific news coverage**. That is the honest finding and it is consistent with the feasibility work:
UK SMEs and micro companies are essentially invisible in news media. It is a result, not a failure,
and the dashboard should present it as one.

In [1]:
import json
import re
from html import unescape
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 170)

DATA = Path(r"C:\Users\visha\Lloyds_Github\data")
PROCESSED = DATA / "processed"
CACHE = DATA / "raw" / "search_cache"

NB08 = PROCESSED / "nb08_spine_news_signals.csv"
NB09 = PROCESSED / "nb09_guardian_signals.csv"
OUT = PROCESSED / "nb14_news_signals_2026-06-30.csv"

SEARCH_DATE = "2026-06-30"    # pinned in NB09, the day treated as "now" for the Guardian run
WINDOW_YEARS = 3
SOURCE = "guardian"

print("cached Guardian responses:", len(list(CACHE.glob("*_guardian.json"))))

cached Guardian responses: 96


## 1. The NB08 output does not vary by company

The first thing to check on any per-company file is whether it actually varies per company. This one
does not. 96 companies produce only six distinct article counts, all bunched at 247 to 249, and
companies that share a count share their sentiment scores exactly. Coverage of 92 out of 96 also sits
against this project's own feasibility finding of 2.8% for the Guardian.

In [2]:
nb08 = pd.read_csv(NB08)
print(f"rows {len(nb08)}, distinct companies {nb08.CompanyNumber.nunique()}")
print("\ndistinct values in the signal columns:")
for c in ["n_articles", "guardian_articles", "gnews_articles", "avg_vader_compound",
          "avg_textblob_polarity", "news_signal"]:
    print(f"  {c:24s} {nb08[c].nunique():>3} distinct   {nb08[c].value_counts().head(3).to_dict()}")

print(f"\nreported coverage: {int(nb08.has_coverage.sum())} / {len(nb08)} "
      f"({100*nb08.has_coverage.mean():.0f}%)")
print("NB07 feasibility measured Guardian coverage at 2.8%.")

rows 96, distinct companies 96

distinct values in the signal columns:
  n_articles                 6 distinct   {249: 31, 248: 30, 247: 29}
  guardian_articles          6 distinct   {249: 31, 248: 30, 247: 29}
  gnews_articles             1 distinct   {0: 96}
  avg_vader_compound         9 distinct   {0.03: 31, 0.078: 22, 0.004: 19}
  avg_textblob_polarity      6 distinct   {0.061: 32, 0.055: 29, 0.072: 29}
  news_signal                7 distinct   {0.854: 31, 0.862: 22, 0.851: 19}

reported coverage: 92 / 96 (96%)
NB07 feasibility measured Guardian coverage at 2.8%.


### Why it happened

NB08 builds one query per template, of the form `"{first three words of search_name}" {template}`.
The `search_name` is not a company name, it is a name plus town plus sector, so the phrase becomes
something like `"whalar london fast"`. That phrase matches nothing. But the Guardian's `q` parameter
treats the remaining template words as OR terms, so `"whalar london fast" funding round` returns
thousands of unrelated articles about funding rounds, capped by `page-size=50`.

Five templates per company at 50 results each, deduplicated, is where 247 to 249 comes from. It is
the same generic Guardian news for every company, which is also why the sentiment is identical.

The cell below demonstrates it live. It needs a Guardian key; without one it prints the counts that
were observed when the audit was run.

In [3]:
import os
KEY = os.environ.get("GUARDIAN_API_KEY", "")

OBSERVED = [('"whalar london fast"', 0, "-"),
            ('"whalar london fast" funding round', 8698,
             "OpenAI, parent firm of ChatGPT, closes $122bn funding round amid AI boom"),
            ('"whalar london fast" investment', 4009,
             "Super League closes in on gamechanging investment from Australia's NRL")]

if KEY:
    import requests, time
    for q, _, _ in OBSERVED:
        p = {"q": q, "from-date": "2025-12-01", "page-size": 50,
             "show-fields": "headline", "api-key": KEY}
        d = requests.get("https://content.guardianapis.com/search", params=p, timeout=25).json()["response"]
        head = d["results"][0]["fields"]["headline"][:72] if d.get("results") else "-"
        print(f"  total={d.get('total'):>6}  q={q}")
        print(f"      first headline: {head}")
        time.sleep(1)
else:
    print("No GUARDIAN_API_KEY set. Counts observed during the audit on 2026-08-14:\n")
    for q, tot, head in OBSERVED:
        print(f"  total={tot:>6}  q={q}")
        print(f"      first headline: {head}")

print("\nThe company phrase alone returns nothing. Adding template words returns thousands of")
print("articles about other companies, which is what NB08 counted as company coverage.")

No GUARDIAN_API_KEY set. Counts observed during the audit on 2026-08-14:

  total=     0  q="whalar london fast"
      first headline: -
  total=  8698  q="whalar london fast" funding round
      first headline: OpenAI, parent firm of ChatGPT, closes $122bn funding round amid AI boom
  total=  4009  q="whalar london fast" investment
      first headline: Super League closes in on gamechanging investment from Australia's NRL

The company phrase alone returns nothing. Adding template words returns thousands of
articles about other companies, which is what NB08 counted as company coverage.


## 2. NB09 is sound, but the saved file predates its own fix

NB09 does the right things: it queries the clean company name only, restricts to `query-fields=body`
and a section filter, caches every raw response, and then verifies each hit before counting it.

Its verification was later hardened three ways: standalone word-boundary matching so "coaster" stops
matching "roller-coaster", a requirement that the name appear in the article's summary fields rather
than buried in the body, and a blocklist for common-word names. The notebook on
`viktor-vishal/unstructured-data-lab` contains all three. The exported CSV does not reflect them, so
it still reports 2 verified companies.

Because every raw Guardian response was cached, verification can be re-derived offline and
deterministically, with no API calls and no risk of a different answer on a different day.

In [4]:
# Verification rules, copied from notebooks/09_guardian_methodology.ipynb on the
# viktor-vishal/unstructured-data-lab branch so the two cannot drift apart silently.
BUSINESS_CONTEXT = ["company", "firm", "business", "ltd", "limited", "plc", "group", "holdings",
    "ceo", "chief executive", "founder", "director", "turnover", "revenue", "profit",
    "acquisition", "merger", "contract", "customers", "employees", "staff", "headquarters"]
UNSEARCHABLE_NAMES = {"coaster", "baroness"}
SECTOR_KEYWORDS = {
    "Manufacturing": ["manufacturing", "factory", "production", "engineering"],
    "Technology, legal & professional": ["technology", "software", "legal", "consultancy", "services"],
    "Fast growth & emerging": ["startup", "funding", "investment", "scale-up", "growth"],
}

_strip = lambda s: unescape(re.sub(r"<[^>]+>", " ", s)).lower()

def _summary_text(a):
    f = a.get("fields", {}) or {}
    return _strip(" ".join([a.get("webTitle", ""), f.get("headline", ""),
                            f.get("standfirst", ""), f.get("trailText", "")]))

def _article_text(a):
    f = a.get("fields", {}) or {}
    return _strip(" ".join([a.get("webTitle", ""), f.get("headline", ""), f.get("standfirst", ""),
                            f.get("trailText", ""), f.get("body", "")]))

def _mentions(text, phrase):
    "Standalone match only, so 'coaster' does not match 'roller-coaster'."
    return re.search(r"(?<![\w-])" + re.escape(phrase) + r"(?![\w-])", text) is not None

def verify_article(a, clean_name, town, sector):
    name = str(clean_name).lower().strip()
    summary, full = _summary_text(a), _article_text(a)
    if name in UNSEARCHABLE_NAMES:
        return False
    if not _mentions(summary, name):        # must be named up front, not in passing
        return False
    town_ok = isinstance(town, str) and len(town) > 1 and _mentions(full, town.lower())
    sector_ok = any(_mentions(full, k) for k in SECTOR_KEYWORDS.get(sector, []))
    context_ok = any(c in full for c in BUSINESS_CONTEXT)
    return town_ok or sector_ok or context_ok

print("verification rules loaded")

verification rules loaded


In [5]:
nb09 = pd.read_csv(NB09, dtype={"CompanyNumber": str})
rows = []
for _, r in nb09.iterrows():
    f = CACHE / f"{r.CompanyNumber}_guardian.json"
    raw, kept, rejected = 0, [], []
    if f.exists():
        resp = json.loads(f.read_text(encoding="utf-8")).get("response", {})
        raw = resp.get("total", 0)
        for a in resp.get("results", []):
            # Title and link together. A headline on its own cannot be checked by a
            # reader; the URL is what makes a claim (or a rejection) verifiable.
            item = (a.get("webTitle", ""), a.get("webUrl", ""))
            (kept if verify_article(a, r.clean_name, r.town, r.sector) else rejected).append(item)
    rows.append({"CompanyNumber": r.CompanyNumber, "raw_hits": raw,
                 "verified": len(kept),
                 "kept_titles": " | ".join(t for t, _ in kept[:3]),
                 "kept_urls": " | ".join(u for _, u in kept[:3]),
                 "rejected_titles": " | ".join(t for t, _ in rejected[:3]),
                 "rejected_urls": " | ".join(u for _, u in rejected[:3])})
rederived = pd.DataFrame(rows)

print(f"companies searched      : {len(rederived)}")
print(f"raw coverage > 0        : {int((rederived.raw_hits > 0).sum())}")
print(f"verified > 0 (corrected): {int((rederived.verified > 0).sum())}")
print(f"verified > 0 (stale CSV): {int((nb09.n_verified > 0).sum())}")

print("\nthe five raw hits, and what they actually were:")
hits = rederived[rederived.raw_hits > 0].merge(
    nb09[["CompanyNumber", "CompanyName", "clean_name"]], on="CompanyNumber")
for _, h in hits.iterrows():
    print(f"\n  {h.CompanyName}  (searched as '{h.clean_name}', {h.raw_hits} raw hits, "
          f"{h.verified} verified)")
    print(f"    rejected: {h.rejected_titles[:150]}")

companies searched      : 96
raw coverage > 0        : 5
verified > 0 (corrected): 0
verified > 0 (stale CSV): 2

the five raw hits, and what they actually were:

  BRS GOLF LIMITED  (searched as 'Brs Golf', 1 raw hits, 0 verified)
    rejected: Cruises, golf and private health: how baby boomer spending has kept UK inflation stubbornly high

  NCC GROUP PLC  (searched as 'Ncc', 3 raw hits, 0 verified)
    rejected: UK manufacturing set for a funding boost to reduce energy costs | AI doom, nature laws and solving the housing problem: five takeaways from day two of

  BARONESS LIMITED  (searched as 'Baroness', 62 raw hits, 0 verified)
    rejected: Michelle Mone says she has ‘no wish’ to remain a Conservative peer | Grooming gangs inquiry ‘must consider ethnicity and religion’, Badenoch says | Lo

  COASTER LIMITED  (searched as 'Coaster', 16 raw hits, 0 verified)
    rejected: Luxury watches, iPads and a Jaguar: what Peter Murrell bought with embezzled funds | Motorhome bought by Murrel

Every raw hit is a name collision: a golf article for BRS GOLF, roller-coaster and bus stories for
COASTER, House of Lords coverage for BARONESS, an unrelated manufacturing piece for NCC, and the
Letby inquiry for ASHTONS. The verification step is doing its job by refusing all of them. Zero
verified coverage is the correct answer, not a broken pipeline.

## 3. The corrected signal file

One row per searched company. The important design choice is that this file distinguishes three
states rather than two, because "we looked and found nothing" and "we never looked" are different
facts and collapsing them would overstate what is known:

- **`searched_no_coverage`**, the 96 companies here
- **`searched_with_coverage`**, currently none
- **`not_searched`**, the other 1,530,998 companies in the universe, which this file simply does not
  contain

`news_rejected_titles` is kept deliberately. It is the evidence that disambiguation ran and refused
the collisions, which is the part of the methodology worth showing.

In [6]:
news = nb09[["CompanyNumber", "CompanyName", "clean_name", "segment", "sector", "town"]].copy()
news = news.merge(rederived, on="CompanyNumber")

news["news_source"] = SOURCE
news["news_search_date"] = SEARCH_DATE
news["news_window_years"] = WINDOW_YEARS
news = news.rename(columns={"raw_hits": "news_raw_hits", "verified": "news_verified_count",
                            "kept_titles": "news_verified_titles",
                            "kept_urls": "news_verified_urls",
                            "rejected_titles": "news_rejected_titles",
                            "rejected_urls": "news_rejected_urls"})
news["news_status"] = news["news_verified_count"].apply(
    lambda v: "searched_with_coverage" if v > 0 else "searched_no_coverage")
# Sentiment is only meaningful where an article survived verification, so it stays null elsewhere
# rather than being written as 0, which would read as a neutral reading that was never taken.
news["news_sentiment_score"] = pd.NA
news["news_sentiment_label"] = news["news_verified_count"].apply(
    lambda v: pd.NA if v > 0 else "no_coverage")

assert news.CompanyNumber.is_unique and len(news) == 96
assert (news.news_verified_count == 0).all(), "a verified article appeared, add FinBERT scoring back"

news.to_csv(OUT, index=False)
print(f"wrote {OUT.name}: {len(news)} rows x {news.shape[1]} columns")
print(news["news_status"].value_counts().to_string())
print("\ncoverage by segment (all zero, shown to make the shape of the sample visible):")
print(news.groupby("segment").agg(companies=("CompanyNumber", "size"),
                                  raw=("news_raw_hits", "sum"),
                                  verified=("news_verified_count", "sum")).to_string())

wrote nb14_news_signals_2026-06-30.csv: 96 rows x 18 columns
news_status
searched_no_coverage    96

coverage by segment (all zero, shown to make the shape of the sample visible):
         companies  raw  verified
segment                          
Large           24    4         0
Medium          24   62         0
Micro           24   16         0
Small           24    1         0


## 4. What the dashboard should say

The news source is now live rather than pending, and what it reports is an absence. Three points
should survive into the write-up:

- **96 companies searched, 0 with verifiable coverage.** Guardian, a three year window, sections
  restricted to business, money, technology and UK news.
- **5 companies had raw hits and all 5 were rejected** as name collisions. That is the
  disambiguation step earning its place, and it is worth showing rather than hiding.
- **The other 1,530,998 companies were never searched**, and the dashboard must not imply otherwise.
  At Guardian's one call per second, searching the full universe would take roughly 18 days of
  continuous requests against a 500 call per day developer quota, which is why a 96 company
  stratified sample was used.

The practical conclusion for the project is the one the feasibility work already reached: news is
not a usable standalone signal for BB and SME prospects, and the Gazette is the source that actually
carries information at this end of the market.